# Fit nonlinear function to network performance

This script takes in the results from data-overview.ipynb, and performs a regression to extract the impact of different regulatory parameters on the data.

In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.backends.backend_pdf import PdfPages

import matplotlib as mpl
import matplotlib.ticker as ticker

import numpy as np
import pandas as pd
import os
import plotly.graph_objects as go
import plotly.express as px
import pickle

# import clustering packages
from sklearn.linear_model import LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline, interp1d
from scipy.signal import savgol_filter

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'
from tqdm import tqdm
from joblib import Parallel, delayed

from stoch_sim_model import *

In [ ]:
# Load data from 2nd infections
reg_model = ''
runs = '-1-'
comment = "sparse-reg" #"prim-Nact-Ediv-vir" #"full-reg-vir" #"act-reg-exp-reg" # mem-reg full_nobl-reg

d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl'

# Import datasets
clustered_mean_df = pd.read_pickle(d_mean)
clustered_mean_df.drop(clustered_mean_df[clustered_mean_df.b_I*clustered_mean_df.S_0 < clustered_mean_df.d_I].index, inplace = True)
with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'rb') as f:
    infection_scenarios = pickle.load(f)

with pd.option_context('display.max_columns', None):
    display(clustered_mean_df)

In [ ]:
# Define parameters
module_reg = {'$N \longrightarrow N^*$': [r for r in Nact_reg if np.std(infection_scenarios[0][r].values) > 0], 
              '$N^* \longrightarrow M$': [r for r in NM_reg if np.std(infection_scenarios[0][r].values) > 0],  
              '$E \longrightarrow M$': [r for r in EM_reg if np.std(infection_scenarios[0][r].values) > 0], 
              '$E \longrightarrow E + E$': [r for r in Ediv_reg if np.std(infection_scenarios[0][r].values) > 0]}
module_reg_labels = {'$N \longrightarrow N^*$': [r for i, r in enumerate(param_names[-16:-12]) if  Nact_reg[i] in module_reg['$N \longrightarrow N^*$']], 
                     '$N^* \longrightarrow M$': [r for i, r in enumerate(param_names[-12:-8]) if  NM_reg[i] in module_reg['$N^* \longrightarrow M$']],
                     '$E \longrightarrow M$': [r for i, r in enumerate(param_names[-8:-4]) if  EM_reg[i] in module_reg['$E \longrightarrow M$']],
                     '$E \longrightarrow E + E$': [r for i, r in enumerate(param_names[-4:]) if  Ediv_reg[i] in module_reg['$E \longrightarrow E + E$']]}

reg = [r for r in Nact_reg + NM_reg + EM_reg + Ediv_reg if np.std(infection_scenarios[0][r].values) > 0]
key_var = 'antigenicity_over_harm'

In [ ]:
# Fit impact of regulatory parameters on performace: marginalizing over modules
for module_index in np.arange(0, len(modules)):
    reg_choice_label = modules[module_index]
    reg_choice = module_reg[reg_choice_label]
    
    x_varname= key_var
    fig_c, axs = plt.subplots(num_categories, len(reg_choice), 
                              figsize=(len(reg_choice)*len(reg_choice), np.maximum(3.5, len(reg_choice))*num_categories), sharex='col', sharey='row')

    data = clustered_mean_df.groupby(reg_choice, as_index = False).mean()
    
    for i, var in enumerate(perf_vars):
        
        for j, p in enumerate(signal_classes):
            data_mean = data.groupby(reg_choice[j], as_index = False).mean()
            im = axs[i, j].scatter(data_mean[reg_choice[j]], data_mean[var], 
                              marker =reg_markers[0], s = 20, edgecolors = 'None')

            y_max = np.max(data[var].values)
            y_min = np.min(data[var].values)

            #if var != 'T_max_pI':
            popt, pcov = curve_fit(sigmoid_1d, data_mean[reg_choice[j]].values, data_mean[var].values, 
                                   p0 =(y_max, y_min, 0.0, 0.1),
                                   method = 'trf', 
                                   bounds = ([np.max(data_mean[var].values), np.minimum(2*y_min, 0), -2*psi_max, -2*psi_max], [2*y_max if y_max > 0 else 0.0, np.min(data_mean[var].values), 2*psi_max, 2*psi_max]),
                                   maxfev=20000)

            sigma_w = np.sqrt(np.diag(pcov))[-1]

            x = np.linspace(-np.max(data_mean[reg_choice[j]]), np.max(data_mean[reg_choice[j]]), 100)
            axs[i, j].plot(x, sigmoid_1d(x, *popt), 'k--')

            axs[i, j].set_title('fit: $y_{max} =$ %5.3f, $y_{min} =$ %5.3f, \n $b=$ %5.3f, $w=$ %5.3f, $\sigma(w) =$ %5.3f' % tuple(popt.tolist() + [sigma_w]), fontsize = 9.0)
                
            axs[i, j].minorticks_off()
            axs[i, j].yaxis.set_tick_params(labelbottom=True)
            axs[i, j].xaxis.set_tick_params(labelbottom=True)
            axs[i, j].tick_params(axis='both', which='major', labelsize=8)
            axs[i, j].set_box_aspect(1)
            axs[i, j].set_ylabel(perf_labels[i], fontsize = 10)
            axs[i, j].set_xlabel(reg_label[module_index*len(reg_choice) + j ], fontsize = 10)
    
    fig_c.subplots_adjust(wspace=.75, hspace=0.5)
    plt.savefig('_figs/'+comment+'-'+reg_choice_label+'-mean_regulation_trends_by_performance'+'.png', dpi=150, bbox_inches="tight")
    del data

In [ ]:
# Regress Performance measures on regulation parameters
num_cpu = 150
variables = reg

def model(input, fit = "linear"):

    out = []
    infection = input[vir_vars].values[0]
    
    for i, perf in enumerate(perf_vars):
        # run regression
        
        if fit == "linear":
            input.loc[:,'dep_var'] = np.log(1 + input[perf].values) if 'Memory' in perf_labels[perf_vars.index(perf)] or 'magnitude' in perf_labels[perf_vars.index(perf)] else input[perf].values
            lr = smf.ols('dep_var ~ ' + ' + '.join(variables), data=input).fit()
    
            # save values
            out.append( (lr.params.values[0: 1 + len(variables)]).tolist() + np.sqrt(np.diag(lr.cov_params())).tolist() + infection.tolist() + [np.mean(input[perf].values), i] )
        else:
            y_max = np.max(input[perf].values)
            y_min = np.min(input[perf].values)
            y_std = np.std(input[perf].values)
            popt, pcov = curve_fit(sigmoid, input[variables].values, input[perf].values, 
                           p0 =[y_max - y_min, 0.0 if y_min > 0.0 else y_min] + len(variables)*[0.0] + len(variables)*[0.0],
                           method = 'trf', 
                           bounds = ([y_max - y_min, np.minimum(2*y_min, 0)] + len(variables)*[-2*psi_max] + len(variables)*[-2.5*psi_max], 
                                     [2*(y_max - y_min) if y_max > 0 else y_std + 0.1, np.maximum(y_min, 0) + y_std + 0.1] + len(variables)*[2*psi_max] + len(variables)*[2.5*psi_max]), 
                                   maxfev=50000)
            
            residuals = input[perf].values - sigmoid(input[variables].values, *popt)
            rmse = np.sqrt(np.mean(residuals**2))
            out.append(popt.tolist() + np.sqrt(np.diag(pcov)).tolist() + infection.tolist() + [np.mean(input[perf].values), np.max(input[perf].values), np.min(input[perf].values), np.median(input[perf].values), i, rmse, y_std, input['harm_pI_noprotection'].values[0]])
    
    return np.vstack(out)

# Compile datasets
impact_data = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(len(virs)/num_cpu),1))(delayed(model)(input = data.groupby(vir_vars + variables, as_index=False).mean(), fit = "nonlinear") 
                                                                                                            for data in infection_scenarios)), 
                               columns = ['y_max', 'y_min'] + [('intercept_' + j) for j in variables] + variables + ['sigma_y_max', 'sigma_y_min'] + [('sigma_intercept_' + j) for j in variables] + ['sigma_'+r  for r in variables] + vir_vars + ['mean_perf', 'max_perf', 'min_perf', 'median_perf', 'perf_index', 'rmse', 'y_std', 'harm_pI_noprotection'])
impact_data['antigenicity_over_harm'] = antigenicity_over_harm(impact_data)

impact_data.to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/'+comment+'-'+'impact_data'+'.pkl')

In [ ]:
# View result
with pd.option_context('display.max_columns', None):
    display(impact_data.loc[(impact_data.perf_index == 0)])